### Notebook to demo spatial aggregation using H3 hexes.

In [1]:
import os

from insert_cursor import insert_df
from pyspark.sql.functions import count, least, lit
from spark_esri import spark_start, spark_stop
from sparkgeo.functions import geo_to_h3, h3_to_geom

### Start a spark instance.

In [2]:
config = {
    "spark.driver.memory": "16G",
    "spark.executor.memory": "16G",
    "spark.jars": os.path.join("C:", os.sep, "sparkgeo", "sparkgeo-3.0.3-0.2.jar"),
}
spark = spark_start(config=config)

### Create cursor from feature layer and make sure coords are in WGS84.

In [3]:
sp_ref = arcpy.SpatialReference(4326)
data = arcpy.da.SearchCursor(
    "Broadcast", ["SHAPE@X", "SHAPE@Y"], spatial_reference=sp_ref
)

### Process the data.

- Create spark dataframe.
- Convert lat/lon to h3 code.
- Group and aggr by h3 code.
- Convert h3 code to polygon.
- Cap count to 1000 max.

In [4]:
df = (
    spark.createDataFrame(data, "x double,y double")
    .withColumn("h3", geo_to_h3("y", "x", lit(9)))
    .groupBy("h3")
    .agg(count("h3").alias("pop"))
    .withColumn("wkb", h3_to_geom("h3"))
    .drop("h3")
    # IMPORTANT: Make sure geometry is the first column.
    .select("wkb", least("pop", lit(1000)).alias("pop"))
)

### Convert spark dataframe to ephemeral feature layer in the ToC.

In [5]:
insert_df(df, "H3", spatial_reference=4326)

### Stop spark instance.

In [ ]:
spark_stop()